### Import Dependencies

In [1]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

In [2]:
os.getenv("LANGSMITH_PROJECT")

'e2e-ai-eng'

In [3]:
import openai
from qdrant_client import QdrantClient
from langsmith import traceable, get_current_run_tree

### Embedding function

In [4]:
@traceable(
    name="embed_query",
    run_type="embedding",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "text-embedding-3-small"
    }
)
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens,
            "total_tokens": response.usage.total_tokens,
        }

    return response.data[0].embedding

### Retrieval function

In [5]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [6]:
@traceable(
    name="retrieve_data",
    run_type="retriever"
)
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

### Format retrieved context function

In [7]:
@traceable(
    name="format_retrieved_context",
    run_type="prompt"
)
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

### Create prompt template function

In [8]:
@traceable(
    name="build_prompt",
    run_type="prompt"
)
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

### Generate answer function

In [9]:
@traceable(
    name="generate_answer",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-nano"
    }
)
def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens,
        }

    return response.choices[0].message.content

### Combined RAG pipeline

In [10]:
@traceable(
    name="rag_pipeline",
)
def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return answer
    

In [11]:
print(rag_pipeline("Do you have a USB connectable fan for hot summers?"))

Yes. We have a couple of USB connectable fans that work well for hot summers:

1) Marame 120mm 5V USB Powered Fan with Speed Controller (120mm x 120mm x 55mm)
- USB-powered via a 3.3 ft USB cable
- Speed control switch with off/low/medium/high for adjustable airflow and noise

2) HZD Desk Fan Rechargeable / Mini Portable Fan (USB-powered; note: this one is USB-powered)
- USB-powered via a 4.9 ft USB cable (works with USB ports like laptop, power bank, AC adapter, car charger)
- 3 speeds (low/medium/high)
- Quiet operation (noise stated as under 50 dB)

If you tell me whether you want something for a desktop (small/portable) or for cooling a device cabinet/router/TV box (larger 120mm), I can recommend the best match.


In [ ]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have above 4 rating.", 10))

In [ ]:
print(rag_pipeline("Could you suggest me some earphones? I am only interested in the ones that have bellow 4 rating.", 10))